# LangChain Summarization & Rule Extraction

This notebook implements an **auditable rule extraction system** for histopathology literature:

1. **Load Disease-Related Texts** - Read UMLS entity files from `test_results_50_docs/relevant_texts/`
2. **Generate Summaries** - Use LangChain with LLM to create concise summaries
3. **Extract Clinical Rules** - Identify actionable rules and patterns from summaries
4. **Maintain Traceability** - Link rules → summaries → sentences → documents
5. **Export Results** - Save with full audit trail for verification

## Key Features

- ✅ **Auditable**: Full traceability from extracted rules back to source text
- ✅ **Structured Output**: JSON format with metadata and provenance
- ✅ **Batch Processing**: Handles multiple UMLS concepts efficiently
- ✅ **LLM-Powered**: Uses state-of-the-art language models via LangChain

## Setup and Configuration

In [5]:
import os
import json
import sys
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime
from collections import defaultdict

# Configuration
LANGCHAIN_DIR = Path.cwd()
# Use JSON files for full database provenance (PMCID, text_element_id)
INPUT_JSON_DIR = LANGCHAIN_DIR / "test_results_50_docs" / "umls_entities"
# Legacy TXT files (less provenance data)
INPUT_TXT_DIR = LANGCHAIN_DIR / "test_results_50_docs" / "relevant_texts"
OUTPUT_DIR = LANGCHAIN_DIR / "summarization_results"
SUMMARIES_DIR = OUTPUT_DIR / "summaries"
RULES_DIR = OUTPUT_DIR / "rules"
AUDIT_DIR = OUTPUT_DIR / "audit_trails"

# Create output directories
for directory in [OUTPUT_DIR, SUMMARIES_DIR, RULES_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Input JSON (with DB provenance): {INPUT_JSON_DIR}")
print(f"Input TXT (legacy):              {INPUT_TXT_DIR}")
print(f"Output directory:                {OUTPUT_DIR}")
print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Input JSON (with DB provenance): /Users/emir/Documents/GitHub/nlp-histo/langchain-summarization/test_results_50_docs/umls_entities
Input TXT (legacy):              /Users/emir/Documents/GitHub/nlp-histo/langchain-summarization/test_results_50_docs/relevant_texts
Output directory:                /Users/emir/Documents/GitHub/nlp-histo/langchain-summarization/summarization_results

Started at: 2026-01-20 11:41:39


## Install Required Packages

Uncomment and run if needed:

In [ ]:
# !pip install langchain langchain-openai python-dotenv tiktoken
# !pip install -U langchain langchain-openai langchain-core

## Import LangChain Components

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Load environment variables (API keys)
load_dotenv()

# Verify API key is set
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  Warning: OPENAI_API_KEY not found in environment")
    print("Please set it in .env file or export OPENAI_API_KEY=your_key")
else:
    print("✅ OpenAI API key loaded successfully")

✅ OpenAI API key loaded successfully


## Load Disease-Related Text Files

Load all TXT files from the `relevant_texts/` directory. Each file represents a UMLS concept with all related sentences.

In [ ]:
def load_text_files(input_dir: Path, limit: Optional[int] = None) -> List[Dict]:
    """
    Load disease-related text files from directory.
    
    Args:
        input_dir: Directory containing TXT files
        limit: Optional limit on number of files to load
    
    Returns:
        List of dicts with file metadata and content
    """
    txt_files = sorted(input_dir.glob("*.txt"))
    
    if limit:
        txt_files = txt_files[:limit]
    
    loaded_files = []
    
    for txt_file in txt_files:
        try:
            # Parse filename: C0001234_Disease_Name.txt
            filename = txt_file.stem
            parts = filename.split('_', 1)
            cui = parts[0]
            concept_name = parts[1].replace('_', ' ') if len(parts) > 1 else cui
            
            # Read file content
            with open(txt_file, 'r', encoding='utf-8') as f:
                content = f.read()
            
            # Parse header information
            lines = content.split('\n')
            metadata = {}
            text_start_idx = 0
            
            for i, line in enumerate(lines[:10]):
                if line.startswith('UMLS CUI:'):
                    metadata['umls_cui'] = line.split(':', 1)[1].strip()
                elif line.startswith('Canonical Name:'):
                    metadata['canonical_name'] = line.split(':', 1)[1].strip()
                elif line.startswith('Entity Label:'):
                    metadata['entity_label'] = line.split(':', 1)[1].strip()
                elif line.startswith('Total Occurrences:'):
                    metadata['total_occurrences'] = int(line.split(':', 1)[1].strip())
                elif line.startswith('Unique Sentences:'):
                    metadata['unique_sentences'] = int(line.split(':', 1)[1].strip())
                elif line.startswith('Text Variants:'):
                    metadata['text_variants'] = line.split(':', 1)[1].strip()
                elif '=' * 50 in line:
                    text_start_idx = i + 1
                    break
            
            # Extract sentences (after header)
            sentences_text = '\n'.join(lines[text_start_idx:]).strip()
            sentences = [s.strip() for s in sentences_text.split('\n\n') if s.strip()]
            
            loaded_files.append({
                'cui': cui,
                'concept_name': concept_name,
                'filename': txt_file.name,
                'filepath': str(txt_file),
                'metadata': metadata,
                'sentences': sentences,
                'full_text': content,
                'num_sentences': len(sentences)
            })
        
        except Exception as e:
            print(f"Warning: Could not load {txt_file.name}: {e}")
    
    return loaded_files


# Load files (start with first 10 for testing)
print("Loading text files...")
text_files = load_text_files(INPUT_TXT_DIR, limit=10)

print(f"\n✅ Loaded {len(text_files)} text files")
print("\nSample files:")
for i, file in enumerate(text_files[:5], 1):
    print(f"  {i}. {file['cui']} - {file['concept_name'][:50]} ({file['num_sentences']} sentences)")

Loading text files...

✅ Loaded 10 text files

Sample files:
  1. C0000737 - Abdominal Pain (2 sentences)
  2. C0000833 - Abscess (4 sentences)
  3. C0000887 - Acantholysis (2 sentences)
  4. C0000921 - Accidental Falls (1 sentences)
  5. C0000925 - Incised wound (1 sentences)


In [ ]:
def load_json_files_with_provenance(input_dir: Path, limit: Optional[int] = None) -> List[Dict]:
    """
    Load UMLS entity JSON files with FULL DATABASE PROVENANCE.
    
    Each sentence includes:
    - pmcid: Links to documents.pmcid in database
    - text_element_id: Links to text_elements.id in database
    - section: Section context from text_elements.path_string
    - entity_text, start_char, end_char: Entity position
    - umls_score: Entity linking confidence
    
    This provides complete traceability back to the database schema.
    
    Args:
        input_dir: Directory containing JSON files
        limit: Optional limit on number of files to load
    
    Returns:
        List of dicts with full provenance metadata
    """
    json_files = sorted(input_dir.glob("*.json"))
    
    if limit:
        json_files = json_files[:limit]
    
    loaded_files = []
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata
            cui = data.get('umls_cui', '')
            concept_name = data.get('canonical_name', '')
            entity_label = data.get('entity_label', '')
            
            # Extract sentences with full provenance
            sentences_with_provenance = []
            for sent_data in data.get('sentences', []):
                sentences_with_provenance.append({
                    'pmcid': sent_data.get('pmcid'),  # Links to documents.pmcid
                    'text_element_id': sent_data.get('text_element_id'),  # Links to text_elements.id
                    'sentence': sent_data.get('sentence'),
                    'section': sent_data.get('section'),
                    'entity_text': sent_data.get('entity_text'),
                    'start_char': sent_data.get('start_char'),
                    'end_char': sent_data.get('end_char'),
                    'umls_score': sent_data.get('umls_score')
                })
            
            # Group by PMCID for statistics
            pmcids = set(s['pmcid'] for s in sentences_with_provenance if s['pmcid'])
            
            loaded_files.append({
                'cui': cui,
                'concept_name': concept_name,
                'entity_label': entity_label,
                'filename': json_file.name,
                'filepath': str(json_file),
                'sentences_with_provenance': sentences_with_provenance,
                # Simple sentence list for compatibility
                'sentences': [s['sentence'] for s in sentences_with_provenance],
                'num_sentences': len(sentences_with_provenance),
                'num_documents': len(pmcids),
                'pmcids': list(pmcids),
                'metadata': {
                    'umls_cui': cui,
                    'canonical_name': concept_name,
                    'entity_label': entity_label,
                    'total_occurrences': data.get('total_occurrences', 0),
                    'unique_entity_texts': data.get('unique_entity_texts', [])
                }
            })
        
        except Exception as e:
            print(f"Warning: Could not load {json_file.name}: {e}")
    
    return loaded_files


# Load JSON files with full provenance
print("Loading JSON files with database provenance...")
json_files = load_json_files_with_provenance(INPUT_JSON_DIR, limit=10)

print(f"\nLoaded {len(json_files)} concept files with DB provenance")
print("\nSample file structure:")
if json_files:
    sample = json_files[0]
    print(f"  CUI: {sample['cui']}")
    print(f"  Concept: {sample['concept_name']}")
    print(f"  Sentences: {sample['num_sentences']}")
    print(f"  Documents (PMCIDs): {sample['num_documents']}")
    if sample['sentences_with_provenance']:
        sent = sample['sentences_with_provenance'][0]
        print(f"\n  Sample sentence provenance:")
        print(f"    PMCID: {sent['pmcid']} -> documents.pmcid")
        print(f"    text_element_id: {sent['text_element_id']} -> text_elements.id")
        print(f"    section: {sent['section']}")
        print(f"    umls_score: {sent['umls_score']}")

Loading JSON files with database provenance...

Loaded 10 concept files with DB provenance

Sample file structure:
  CUI: C0000368
  Concept: 3,3'-Diaminobenzidine
  Sentences: 1
  Documents (PMCIDs): 1

  Sample sentence provenance:
    PMCID: PMC11503264 -> documents.pmcid
    text_element_id: 446 -> text_elements.id
    section: 2. Materials and Methods
    umls_score: 0.7855776
[{'cui': 'C0000368', 'concept_name': "3,3'-Diaminobenzidine", 'entity_label': 'ENTITY', 'filename': 'C0000368_3_3_-Diaminobenzidine.json', 'filepath': '/Users/emir/Documents/GitHub/nlp-histo/langchain-summarization/test_results_50_docs/umls_entities/C0000368_3_3_-Diaminobenzidine.json', 'sentences_with_provenance': [{'pmcid': 'PMC11503264', 'text_element_id': 446, 'sentence': "The immunohistochemical study (IHC) was performed in an automated Ventana machine on 3-micron-thick tissue sections cut from the paraffin blocks. Briefly, the sections were deparaffinized and dehydrated. Ethylene-diamine tetra-acetic a

## Define LangChain Prompts

Create specialized prompts for summarization and rule extraction.

In [ ]:
SUMMARIZATION_PROMPT = """You are a medical expert analyzing histopathology literature.

Concept: {concept_name}

Text:
{text}

Task: Create a concise, accurate summary of the key findings about this medical concept. Focus on:
- Clinical significance and diagnostic criteria
- Histopathological features and characteristics
- Treatment approaches and outcomes
- Important associations or risk factors

Keep the summary factual, specific, and medically accurate. Limit to 200-300 words.

Summary:"""

RULE_EXTRACTION_PROMPT = """You are a medical knowledge engineer extracting clinical rules from literature.

Concept: {concept_name}

Summary:
{summary}

Task: Extract structured clinical rules from this summary. For each rule, provide:
1. Rule Type (e.g., "Diagnostic Criterion", "Treatment Guideline", "Risk Factor", "Prognostic Indicator")
2. Condition (when does this rule apply?)
3. Action/Conclusion (what should be done or concluded?)
4. Confidence Level (High/Medium/Low based on language in the summary)

Format each rule as:
RULE [N]:
Type: [rule type]
Condition: [specific condition]
Action: [specific action or conclusion]
Confidence: [High/Medium/Low]
---

Only extract rules that are explicitly supported by the summary. If no clear rules exist, respond with "No extractable rules found."

Extracted Rules:"""

SUMMARIZATION_TEMPLATE = ChatPromptTemplate(
    [('system', SUMMARIZATION_PROMPT)]
)

RULE_EXTRACTION_TEMPLATE = ChatPromptTemplate(
    [('system', RULE_EXTRACTION_PROMPT)]
)


# Rule extraction prompt


print("✅ Prompts defined")

TypeError: ChatPromptTemplate.__init__() missing 1 required positional argument: 'messages'

## Initialize LLM and Chains

In [ ]:
# Initialize LLM (using GPT-4 for better quality)
llm = ChatOpenAI(
    model="gpt-4",
    temperature=0.3,  # Lower temperature for more consistent output
    max_tokens=1000
)

# Create chains
summarization_chain = LLMChain(llm=llm, prompt=SUMMARIZATION_PROMPT)
rule_extraction_chain = LLMChain(llm=llm, prompt=RULE_EXTRACTION_PROMPT)

print("✅ LLM and chains initialized")
print(f"Model: {llm.model_name}")
print(f"Temperature: {llm.temperature}")

## Process Documents: Summarization & Rule Extraction

Process each UMLS concept through the pipeline with full traceability.

In [ ]:
def process_document(file_data: Dict, summarization_chain, rule_extraction_chain) -> Dict:
    """
    Process a single document through summarization and rule extraction.
    
    Returns:
        Dict with summary, rules, and audit trail
    """
    cui = file_data['cui']
    concept_name = file_data['concept_name']
    
    # Combine sentences for summarization (limit to first 3000 chars to avoid token limits)
    combined_text = '\n\n'.join(file_data['sentences'])
    if len(combined_text) > 3000:
        combined_text = combined_text[:3000] + "\n\n[Text truncated for length...]"
    
    try:
        # Step 1: Generate summary
        summary_result = summarization_chain.invoke({
            "concept_name": concept_name,
            "text": combined_text
        })
        summary = summary_result['text'].strip()
        
        # Step 2: Extract rules from summary
        rules_result = rule_extraction_chain.invoke({
            "concept_name": concept_name,
            "summary": summary
        })
        rules_text = rules_result['text'].strip()
        
        # Parse rules into structured format
        parsed_rules = parse_rules(rules_text)
        
        # Build audit trail
        audit_trail = {
            'cui': cui,
            'concept_name': concept_name,
            'source_file': file_data['filename'],
            'num_source_sentences': file_data['num_sentences'],
            'metadata': file_data['metadata'],
            'processing_timestamp': datetime.now().isoformat(),
            'summary': summary,
            'extracted_rules': parsed_rules,
            'num_rules': len(parsed_rules),
            'source_sentences': file_data['sentences'][:10],  # Store first 10 for verification
            'traceability': {
                'rules_to_summary': 'All rules extracted from summary',
                'summary_to_sentences': 'Summary generated from all sentences in file',
                'sentences_to_documents': 'Sentences linked to PMCIDs in original data'
            }
        }
        
        return {
            'status': 'success',
            'cui': cui,
            'concept_name': concept_name,
            'summary': summary,
            'rules': parsed_rules,
            'audit_trail': audit_trail
        }
    
    except Exception as e:
        return {
            'status': 'error',
            'cui': cui,
            'concept_name': concept_name,
            'error': str(e)
        }


def parse_rules(rules_text: str) -> List[Dict]:
    """
    Parse structured rules from LLM output.
    """
    rules = []
    
    if "No extractable rules found" in rules_text:
        return rules
    
    # Split by rule separator
    rule_blocks = rules_text.split('RULE ')
    
    for block in rule_blocks[1:]:  # Skip first empty split
        try:
            lines = block.strip().split('\n')
            rule = {'raw_text': block.strip()}
            
            for line in lines:
                line = line.strip()
                if line.startswith('Type:'):
                    rule['type'] = line.split(':', 1)[1].strip()
                elif line.startswith('Condition:'):
                    rule['condition'] = line.split(':', 1)[1].strip()
                elif line.startswith('Action:'):
                    rule['action'] = line.split(':', 1)[1].strip()
                elif line.startswith('Confidence:'):
                    rule['confidence'] = line.split(':', 1)[1].strip()
            
            if 'type' in rule and 'action' in rule:
                rules.append(rule)
        
        except Exception as e:
            print(f"Warning: Could not parse rule: {e}")
    
    return rules


print("✅ Processing functions defined")

## Run Processing Pipeline

Process all loaded documents through summarization and rule extraction.

In [ ]:
results = []
errors = []

print("="*80)
print("Starting Processing Pipeline")
print("="*80 + "\n")

for i, file_data in enumerate(text_files, 1):
    cui = file_data['cui']
    concept_name = file_data['concept_name']
    
    print(f"[{i}/{len(text_files)}] Processing {cui} - {concept_name[:50]}...")
    
    result = process_document(file_data, summarization_chain, rule_extraction_chain)
    
    if result['status'] == 'success':
        results.append(result)
        print(f"  ✅ Summary: {len(result['summary'])} chars, Rules: {len(result['rules'])}")
    else:
        errors.append(result)
        print(f"  ❌ Error: {result['error']}")

print("\n" + "="*80)
print(f"Processing Complete: {len(results)} successful, {len(errors)} errors")
print("="*80)

## Save Results

Save summaries, rules, and audit trails to separate files.

In [ ]:
print("Saving results...\n")

for result in results:
    cui = result['cui']
    concept_name = result['concept_name'].replace(' ', '_')[:50]
    
    # Save summary
    summary_file = SUMMARIES_DIR / f"{cui}_{concept_name}_summary.txt"
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(f"UMLS CUI: {cui}\n")
        f.write(f"Concept: {result['concept_name']}\n")
        f.write("="*80 + "\n\n")
        f.write(result['summary'])
    
    # Save rules (if any)
    if result['rules']:
        rules_file = RULES_DIR / f"{cui}_{concept_name}_rules.json"
        with open(rules_file, 'w', encoding='utf-8') as f:
            json.dump({
                'cui': cui,
                'concept_name': result['concept_name'],
                'num_rules': len(result['rules']),
                'rules': result['rules']
            }, f, indent=2, ensure_ascii=False)
    
    # Save audit trail
    audit_file = AUDIT_DIR / f"{cui}_{concept_name}_audit.json"
    with open(audit_file, 'w', encoding='utf-8') as f:
        json.dump(result['audit_trail'], f, indent=2, ensure_ascii=False)

# Save master index
index_file = OUTPUT_DIR / "processing_index.json"
with open(index_file, 'w', encoding='utf-8') as f:
    json.dump({
        'processing_date': datetime.now().isoformat(),
        'total_processed': len(results),
        'total_errors': len(errors),
        'total_rules_extracted': sum(len(r['rules']) for r in results),
        'processed_concepts': [
            {
                'cui': r['cui'],
                'concept_name': r['concept_name'],
                'num_rules': len(r['rules']),
                'has_summary': True
            }
            for r in results
        ]
    }, f, indent=2)

print(f"✅ Saved {len(results)} summaries to {SUMMARIES_DIR}")
print(f"✅ Saved {sum(1 for r in results if r['rules'])} rule files to {RULES_DIR}")
print(f"✅ Saved {len(results)} audit trails to {AUDIT_DIR}")
print(f"✅ Saved master index to {index_file}")

## Display Sample Results

In [ ]:
if results:
    sample = results[0]
    
    print("="*80)
    print("SAMPLE RESULT")
    print("="*80)
    print(f"\nConcept: {sample['concept_name']}")
    print(f"CUI: {sample['cui']}")
    print("\n" + "-"*80)
    print("SUMMARY:")
    print("-"*80)
    print(sample['summary'])
    
    if sample['rules']:
        print("\n" + "-"*80)
        print(f"EXTRACTED RULES ({len(sample['rules'])}):")
        print("-"*80)
        for i, rule in enumerate(sample['rules'], 1):
            print(f"\nRule {i}:")
            print(f"  Type: {rule.get('type', 'N/A')}")
            print(f"  Condition: {rule.get('condition', 'N/A')}")
            print(f"  Action: {rule.get('action', 'N/A')}")
            print(f"  Confidence: {rule.get('confidence', 'N/A')}")
    else:
        print("\n(No rules extracted)")
    
    print("\n" + "="*80)

## Statistics Summary

In [ ]:
print("="*80)
print("PROCESSING STATISTICS")
print("="*80)

total_rules = sum(len(r['rules']) for r in results)
concepts_with_rules = sum(1 for r in results if r['rules'])

print(f"\nDocuments processed: {len(results)}")
print(f"Total rules extracted: {total_rules}")
print(f"Concepts with rules: {concepts_with_rules}")
print(f"Average rules per concept: {total_rules / len(results) if results else 0:.1f}")

if total_rules > 0:
    # Rule type distribution
    rule_types = defaultdict(int)
    for result in results:
        for rule in result['rules']:
            rule_type = rule.get('type', 'Unknown')
            rule_types[rule_type] += 1
    
    print("\nRule Type Distribution:")
    for rule_type, count in sorted(rule_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  • {rule_type}: {count}")

print("\n" + "="*80)
print("✅ Pipeline completed successfully!")
print("="*80)

## Verify Traceability

Demonstrate the audit trail from rules back to source text.

In [ ]:
if results and results[0]['rules']:
    sample = results[0]
    audit = sample['audit_trail']
    
    print("="*80)
    print("TRACEABILITY DEMONSTRATION")
    print("="*80)
    print(f"\nConcept: {audit['concept_name']}")
    print(f"CUI: {audit['cui']}")
    
    if audit['extracted_rules']:
        rule = audit['extracted_rules'][0]
        print("\n" + "-"*80)
        print("RULE:")
        print("-"*80)
        print(f"Type: {rule.get('type', 'N/A')}")
        print(f"Action: {rule.get('action', 'N/A')}")
        
        print("\n" + "-"*80)
        print("TRACED TO SUMMARY:")
        print("-"*80)
        print(audit['summary'][:300] + "...")
        
        print("\n" + "-"*80)
        print("TRACED TO SOURCE SENTENCES:")
        print("-"*80)
        for i, sent in enumerate(audit['source_sentences'][:3], 1):
            print(f"{i}. {sent[:150]}..." if len(sent) > 150 else f"{i}. {sent}")
        
        print("\n" + "-"*80)
        print("TRACED TO SOURCE FILE:")
        print("-"*80)
        print(f"File: {audit['source_file']}")
        print(f"Total sentences: {audit['num_source_sentences']}")
        print(f"Processing timestamp: {audit['processing_timestamp']}")
    
    print("\n" + "="*80)
    print("✅ Full traceability maintained: Rule → Summary → Sentences → Documents")
    print("="*80)
else:
    print("No rules with traceability to demonstrate")

---

# Advanced Summarization Strategies

The approaches below use LangChain's built-in summarization chains for handling longer documents:

1. **Map-Reduce**: Processes each chunk independently (map step), then combines all summaries into a final summary (reduce step). Good for parallel processing.

2. **Refine**: Iteratively refines the summary by going through each chunk sequentially. Better for maintaining context and adding information incrementally (e.g., figures/tables added later).

## Setup for Advanced Summarization

Import additional LangChain components for map-reduce and refine strategies.

In [ ]:
# Load ALL JSON files from 50 papers with FULL DATABASE PROVENANCE
print("Loading all JSON files from 50 papers (with database provenance)...")
all_text_files = load_json_files_with_provenance(INPUT_JSON_DIR, limit=None)

print(f"\nLoaded {len(all_text_files)} UMLS concept files")

# Show statistics
total_sentences = sum(f['num_sentences'] for f in all_text_files)
total_pmcids = len(set(pmcid for f in all_text_files for pmcid in f['pmcids']))
print(f"Total sentences across all files: {total_sentences}")
print(f"Total unique PMCIDs (database documents): {total_pmcids}")
print(f"Average sentences per concept: {total_sentences / len(all_text_files):.1f}")

# Show top concepts by sentence count
sorted_by_sentences = sorted(all_text_files, key=lambda x: x['num_sentences'], reverse=True)
print("\nTop 10 concepts by sentence count:")
for i, f in enumerate(sorted_by_sentences[:10], 1):
    print(f"  {i}. {f['cui']} - {f['concept_name'][:40]} ({f['num_sentences']} sentences, {f['num_documents']} docs)")

print("\n" + "="*60)
print("DATABASE PROVENANCE AVAILABLE:")
print("  - pmcid -> documents.pmcid")
print("  - text_element_id -> text_elements.id")
print("  - section -> text_elements.path_string")
print("="*60)

In [ ]:
# Load ALL text files from 50 papers (no limit)
print("Loading all text files from 50 papers...")
all_text_files = load_text_files(INPUT_DIR, limit=None)

print(f"\nLoaded {len(all_text_files)} UMLS concept files")

# Show statistics
total_sentences = sum(f['num_sentences'] for f in all_text_files)
print(f"Total sentences across all files: {total_sentences}")
print(f"Average sentences per concept: {total_sentences / len(all_text_files):.1f}")

# Show top concepts by sentence count
sorted_by_sentences = sorted(all_text_files, key=lambda x: x['num_sentences'], reverse=True)
print("\nTop 10 concepts by sentence count:")
for i, f in enumerate(sorted_by_sentences[:10], 1):
    print(f"  {i}. {f['cui']} - {f['concept_name'][:40]} ({f['num_sentences']} sentences)")

## Strategy 1: Map-Reduce Summarization

**How it works:**
1. **Map step**: Each chunk of text is summarized independently
2. **Reduce step**: All chunk summaries are combined into a final summary

**Advantages:**
- Parallelizable (chunks can be processed simultaneously)
- Works well for large documents
- Good when chunks are relatively independent

**Use case:** When you have all text ready at once and want efficient batch processing.

In [ ]:
# Define custom prompts for Map-Reduce summarization
MAP_PROMPT = PromptTemplate(
    input_variables=["text"],
    template="""You are a medical expert analyzing histopathology literature.

Text chunk:
{text}

Summarize the key medical findings from this text chunk. Focus on:
- Clinical observations and diagnoses
- Histopathological features
- Treatment details and outcomes
- Risk factors and associations

Keep the summary concise and factual.

Summary:"""
)

COMBINE_PROMPT = PromptTemplate(
    input_variables=["text"],
    template="""You are a medical expert synthesizing histopathology literature.

The following are summaries from different parts of a medical document:

{text}

Create a comprehensive, unified summary that:
- Integrates all key findings without repetition
- Maintains medical accuracy
- Highlights the most clinically relevant information
- Is organized logically (diagnosis, features, treatment, outcomes)

Limit to 300-400 words.

Final Summary:"""
)

# Create map-reduce chain
map_reduce_chain = load_summarize_chain(
    llm=llm,
    chain_type="map_reduce",
    map_prompt=MAP_PROMPT,
    combine_prompt=COMBINE_PROMPT,
    verbose=False
)

print("Map-Reduce chain configured")

In [ ]:
def process_with_map_reduce(file_data: Dict, chain, text_splitter) -> Dict:
    """
    Process a single document using map-reduce summarization.
    
    Args:
        file_data: Dict containing sentences and metadata
        chain: Map-reduce summarization chain
        text_splitter: Text splitter for chunking
    
    Returns:
        Dict with summary and metadata
    """
    cui = file_data['cui']
    concept_name = file_data['concept_name']
    
    # Combine all sentences
    combined_text = '\n\n'.join(file_data['sentences'])
    
    # Skip if too short (no need for chunking)
    if len(combined_text) < 500:
        return {
            'status': 'skipped',
            'cui': cui,
            'concept_name': concept_name,
            'reason': 'Text too short for map-reduce'
        }
    
    try:
        # Split into chunks and create LangChain Documents
        chunks = text_splitter.split_text(combined_text)
        docs = [LCDocument(page_content=chunk) for chunk in chunks]
        
        # Run map-reduce chain
        result = chain.invoke(docs)
        summary = result['output_text'].strip()
        
        return {
            'status': 'success',
            'cui': cui,
            'concept_name': concept_name,
            'summary': summary,
            'num_chunks': len(chunks),
            'source_file': file_data['filename'],
            'num_sentences': file_data['num_sentences'],
            'metadata': file_data['metadata']
        }
    
    except Exception as e:
        return {
            'status': 'error',
            'cui': cui,
            'concept_name': concept_name,
            'error': str(e)
        }

print("Map-reduce processing function defined")

In [ ]:
# Run Map-Reduce summarization on all files
map_reduce_results = []
map_reduce_errors = []
map_reduce_skipped = []

print("="*80)
print("MAP-REDUCE SUMMARIZATION")
print("="*80 + "\n")

for i, file_data in enumerate(all_text_files, 1):
    cui = file_data['cui']
    concept_name = file_data['concept_name']
    
    if i % 50 == 0 or i <= 5:
        print(f"[{i}/{len(all_text_files)}] Processing {cui} - {concept_name[:40]}...")
    
    result = process_with_map_reduce(file_data, map_reduce_chain, text_splitter)
    
    if result['status'] == 'success':
        map_reduce_results.append(result)
    elif result['status'] == 'skipped':
        map_reduce_skipped.append(result)
    else:
        map_reduce_errors.append(result)
        print(f"  Error on {cui}: {result.get('error', 'Unknown')}")

print(f"\n{'='*80}")
print(f"Map-Reduce Complete:")
print(f"  Successful: {len(map_reduce_results)}")
print(f"  Skipped (too short): {len(map_reduce_skipped)}")
print(f"  Errors: {len(map_reduce_errors)}")
print("="*80)

In [ ]:
# Save Map-Reduce results
print("Saving Map-Reduce summaries...")

for result in map_reduce_results:
    cui = result['cui']
    concept_name = result['concept_name'].replace(' ', '_')[:50]
    
    # Save summary
    summary_file = MAPREDUCE_DIR / f"{cui}_{concept_name}_mapreduce.txt"
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(f"UMLS CUI: {cui}\n")
        f.write(f"Concept: {result['concept_name']}\n")
        f.write(f"Strategy: Map-Reduce\n")
        f.write(f"Chunks processed: {result['num_chunks']}\n")
        f.write("="*80 + "\n\n")
        f.write(result['summary'])

# Save index
mapreduce_index = MAPREDUCE_DIR / "mapreduce_index.json"
with open(mapreduce_index, 'w', encoding='utf-8') as f:
    json.dump({
        'strategy': 'map_reduce',
        'processing_date': datetime.now().isoformat(),
        'total_processed': len(map_reduce_results),
        'total_skipped': len(map_reduce_skipped),
        'total_errors': len(map_reduce_errors),
        'summaries': [
            {
                'cui': r['cui'],
                'concept_name': r['concept_name'],
                'num_chunks': r['num_chunks'],
                'num_sentences': r['num_sentences']
            }
            for r in map_reduce_results
        ]
    }, f, indent=2)

print(f"Saved {len(map_reduce_results)} map-reduce summaries to {MAPREDUCE_DIR}")

# Show sample
if map_reduce_results:
    sample = map_reduce_results[0]
    print(f"\nSample Map-Reduce Summary ({sample['cui']} - {sample['concept_name'][:30]}):")
    print("-"*60)
    print(sample['summary'][:500] + "..." if len(sample['summary']) > 500 else sample['summary'])

## Strategy 2: Refine Summarization

**How it works:**
1. **Initial summary**: Create a summary from the first chunk
2. **Iterative refinement**: For each subsequent chunk, refine the existing summary by incorporating new information

**Advantages:**
- Maintains context across all chunks
- Good for sequential/incremental data (e.g., adding figure/table info later)
- Produces more coherent summaries for connected content
- Better for documents where later sections build on earlier content

**Use case:** When information is available incrementally or needs to be integrated with existing context (e.g., adding figures and tables information after initial text processing).

In [ ]:
# Define custom prompts for Refine summarization
REFINE_INITIAL_PROMPT = PromptTemplate(
    input_variables=["text"],
    template="""You are a medical expert analyzing histopathology literature.

Text:
{text}

Create a comprehensive medical summary focusing on:
- Clinical observations and diagnoses
- Histopathological features and characteristics
- Treatment approaches and outcomes
- Risk factors and associations

Keep the summary factual and medically accurate.

Summary:"""
)

REFINE_PROMPT = PromptTemplate(
    input_variables=["existing_answer", "text"],
    template="""You are a medical expert refining a summary with new information.

Existing Summary:
{existing_answer}

New Information:
{text}

Task: Refine and expand the existing summary by incorporating relevant new information.
- Add new findings that complement the existing summary
- Update or correct information if the new text provides more detail
- Maintain coherence and avoid redundancy
- Keep the focus on clinical relevance

If the new information doesn't add value, return the existing summary unchanged.

Refined Summary:"""
)

# Create refine chain
refine_chain = load_summarize_chain(
    llm=llm,
    chain_type="refine",
    question_prompt=REFINE_INITIAL_PROMPT,
    refine_prompt=REFINE_PROMPT,
    verbose=False
)

print("Refine chain configured")

In [ ]:
def process_with_refine(file_data: Dict, chain, text_splitter) -> Dict:
    """
    Process a single document using refine summarization.
    
    The refine strategy is particularly useful when:
    - Information comes incrementally (e.g., figures/tables added later)
    - Context needs to be maintained across chunks
    - Documents have interconnected sections
    
    Args:
        file_data: Dict containing sentences and metadata
        chain: Refine summarization chain
        text_splitter: Text splitter for chunking
    
    Returns:
        Dict with summary and metadata
    """
    cui = file_data['cui']
    concept_name = file_data['concept_name']
    
    # Combine all sentences
    combined_text = '\n\n'.join(file_data['sentences'])
    
    # Skip if too short
    if len(combined_text) < 500:
        return {
            'status': 'skipped',
            'cui': cui,
            'concept_name': concept_name,
            'reason': 'Text too short for refine'
        }
    
    try:
        # Split into chunks and create LangChain Documents
        chunks = text_splitter.split_text(combined_text)
        docs = [LCDocument(page_content=chunk) for chunk in chunks]
        
        # Run refine chain
        result = chain.invoke(docs)
        summary = result['output_text'].strip()
        
        return {
            'status': 'success',
            'cui': cui,
            'concept_name': concept_name,
            'summary': summary,
            'num_chunks': len(chunks),
            'num_refinements': len(chunks) - 1,  # First chunk is initial, rest are refinements
            'source_file': file_data['filename'],
            'num_sentences': file_data['num_sentences'],
            'metadata': file_data['metadata']
        }
    
    except Exception as e:
        return {
            'status': 'error',
            'cui': cui,
            'concept_name': concept_name,
            'error': str(e)
        }

print("Refine processing function defined")

In [ ]:
# Run Refine summarization on all files
refine_results = []
refine_errors = []
refine_skipped = []

print("="*80)
print("REFINE SUMMARIZATION")
print("="*80 + "\n")

for i, file_data in enumerate(all_text_files, 1):
    cui = file_data['cui']
    concept_name = file_data['concept_name']
    
    if i % 50 == 0 or i <= 5:
        print(f"[{i}/{len(all_text_files)}] Processing {cui} - {concept_name[:40]}...")
    
    result = process_with_refine(file_data, refine_chain, text_splitter)
    
    if result['status'] == 'success':
        refine_results.append(result)
    elif result['status'] == 'skipped':
        refine_skipped.append(result)
    else:
        refine_errors.append(result)
        print(f"  Error on {cui}: {result.get('error', 'Unknown')}")

print(f"\n{'='*80}")
print(f"Refine Complete:")
print(f"  Successful: {len(refine_results)}")
print(f"  Skipped (too short): {len(refine_skipped)}")
print(f"  Errors: {len(refine_errors)}")
print("="*80)

In [ ]:
# Save Refine results
print("Saving Refine summaries...")

for result in refine_results:
    cui = result['cui']
    concept_name = result['concept_name'].replace(' ', '_')[:50]
    
    # Save summary
    summary_file = REFINE_DIR / f"{cui}_{concept_name}_refine.txt"
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(f"UMLS CUI: {cui}\n")
        f.write(f"Concept: {result['concept_name']}\n")
        f.write(f"Strategy: Refine\n")
        f.write(f"Chunks processed: {result['num_chunks']}\n")
        f.write(f"Refinement iterations: {result['num_refinements']}\n")
        f.write("="*80 + "\n\n")
        f.write(result['summary'])

# Save index
refine_index = REFINE_DIR / "refine_index.json"
with open(refine_index, 'w', encoding='utf-8') as f:
    json.dump({
        'strategy': 'refine',
        'processing_date': datetime.now().isoformat(),
        'total_processed': len(refine_results),
        'total_skipped': len(refine_skipped),
        'total_errors': len(refine_errors),
        'summaries': [
            {
                'cui': r['cui'],
                'concept_name': r['concept_name'],
                'num_chunks': r['num_chunks'],
                'num_refinements': r['num_refinements'],
                'num_sentences': r['num_sentences']
            }
            for r in refine_results
        ]
    }, f, indent=2)

print(f"Saved {len(refine_results)} refine summaries to {REFINE_DIR}")

# Show sample
if refine_results:
    sample = refine_results[0]
    print(f"\nSample Refine Summary ({sample['cui']} - {sample['concept_name'][:30]}):")
    print("-"*60)
    print(sample['summary'][:500] + "..." if len(sample['summary']) > 500 else sample['summary'])

## Strategy Comparison

Compare the results from both summarization strategies.

In [ ]:
print("="*80)
print("SUMMARIZATION STRATEGY COMPARISON")
print("="*80)

print("\n" + "-"*40)
print("PROCESSING STATISTICS")
print("-"*40)
print(f"\n{'Strategy':<15} {'Successful':<12} {'Skipped':<10} {'Errors':<10}")
print("-"*47)
print(f"{'Map-Reduce':<15} {len(map_reduce_results):<12} {len(map_reduce_skipped):<10} {len(map_reduce_errors):<10}")
print(f"{'Refine':<15} {len(refine_results):<12} {len(refine_skipped):<10} {len(refine_errors):<10}")

# Compare summaries for same concepts
print("\n" + "-"*40)
print("SUMMARY LENGTH COMPARISON")
print("-"*40)

# Build lookup by CUI
mr_by_cui = {r['cui']: r for r in map_reduce_results}
rf_by_cui = {r['cui']: r for r in refine_results}

common_cuis = set(mr_by_cui.keys()) & set(rf_by_cui.keys())
print(f"\nConcepts processed by both strategies: {len(common_cuis)}")

if common_cuis:
    mr_lengths = [len(mr_by_cui[cui]['summary']) for cui in common_cuis]
    rf_lengths = [len(rf_by_cui[cui]['summary']) for cui in common_cuis]
    
    print(f"\nAverage summary length:")
    print(f"  Map-Reduce: {sum(mr_lengths)/len(mr_lengths):.0f} chars")
    print(f"  Refine:     {sum(rf_lengths)/len(rf_lengths):.0f} chars")

# Show side-by-side comparison for one concept
print("\n" + "-"*40)
print("SIDE-BY-SIDE COMPARISON (First Common Concept)")
print("-"*40)

if common_cuis:
    sample_cui = list(common_cuis)[0]
    mr_sample = mr_by_cui[sample_cui]
    rf_sample = rf_by_cui[sample_cui]
    
    print(f"\nConcept: {mr_sample['concept_name']}")
    print(f"CUI: {sample_cui}")
    
    print(f"\n--- MAP-REDUCE ({len(mr_sample['summary'])} chars, {mr_sample['num_chunks']} chunks) ---")
    print(mr_sample['summary'][:400] + "..." if len(mr_sample['summary']) > 400 else mr_sample['summary'])
    
    print(f"\n--- REFINE ({len(rf_sample['summary'])} chars, {rf_sample['num_refinements']} refinements) ---")
    print(rf_sample['summary'][:400] + "..." if len(rf_sample['summary']) > 400 else rf_sample['summary'])

print("\n" + "="*80)
print("RECOMMENDATION:")
print("-"*80)
print("""
- Use MAP-REDUCE when:
  * You have all text available upfront
  * Processing speed is important (parallelizable)
  * Chunks are relatively independent
  
- Use REFINE when:
  * Information comes incrementally (figures/tables added later)
  * Context needs to be maintained across chunks
  * You need to update summaries with new information
  * Document sections build on each other
""")
print("="*80)

## Bonus: Incremental Summarization with Refine

The refine strategy is ideal for incrementally adding information. This example shows how to:
1. Start with a text-based summary
2. Later refine it with figure descriptions
3. Then refine it again with table data

This approach is useful when figures/tables are processed separately or become available later.

In [ ]:
def refine_summary_incrementally(
    initial_summary: str,
    additional_info: str,
    info_type: str = "additional information"
) -> str:
    """
    Refine an existing summary with new information using LLM.
    
    This function demonstrates how to use the refine approach for
    incremental updates - perfect for adding figure/table data later.
    
    Args:
        initial_summary: The existing summary to refine
        additional_info: New information to incorporate (e.g., figure descriptions)
        info_type: Description of the type of information being added
    
    Returns:
        The refined summary
    """
    incremental_prompt = PromptTemplate(
        input_variables=["existing_summary", "new_info", "info_type"],
        template="""You are a medical expert refining a summary with new information.

Existing Summary:
{existing_summary}

{info_type}:
{new_info}

Task: Refine the existing summary by incorporating the new {info_type}.
- Integrate visual findings if figure descriptions are provided
- Add quantitative data if table data is provided
- Maintain coherence with existing content
- Don't repeat information already covered
- Keep the medical accuracy and clinical relevance

Refined Summary:"""
    )
    
    chain = LLMChain(llm=llm, prompt=incremental_prompt)
    result = chain.invoke({
        "existing_summary": initial_summary,
        "new_info": additional_info,
        "info_type": info_type
    })
    
    return result['text'].strip()


# Example: Demonstrate incremental refinement workflow
print("="*80)
print("INCREMENTAL SUMMARIZATION DEMO")
print("="*80)

# Step 1: Start with text-based summary (simulated)
example_text_summary = """
Hepatocellular carcinoma (HCC) is a primary liver malignancy characterized by 
trabecular and pseudoglandular growth patterns. The tumor cells show increased 
nuclear-to-cytoplasmic ratio and prominent nucleoli. Key diagnostic features 
include reticulin loss and CD34 positivity in sinusoidal endothelium.
"""

print("\nStep 1: Initial text-based summary")
print("-"*40)
print(example_text_summary.strip())

# Step 2: Simulate adding figure information
example_figure_info = """
Figure 1: H&E staining at 200x magnification shows well-differentiated HCC with 
clear cell change and focal fatty infiltration. Arrows indicate mitotic figures.

Figure 2: Immunohistochemistry panel showing diffuse AFP positivity (brown staining)
confirming hepatocellular origin. GPC-3 shows membranous and cytoplasmic staining.
"""

print("\nStep 2: Adding figure descriptions...")
print("-"*40)
# In practice, uncomment the following to run the LLM:
# refined_with_figures = refine_summary_incrementally(
#     example_text_summary, 
#     example_figure_info,
#     "Figure descriptions"
# )
# print(refined_with_figures)
print("[LLM call would refine summary with figure descriptions here]")

# Step 3: Simulate adding table data
example_table_info = """
Table 1: Patient cohort characteristics (n=50)
- Mean age: 62.4 years (SD 8.3)
- Male/Female ratio: 3.2:1
- Cirrhosis present: 78%
- AFP > 400 ng/mL: 45%
- Tumor size: 4.2 cm (range 1.5-12.8 cm)

Table 2: Survival outcomes
- 1-year survival: 68%
- 3-year survival: 42%
- Recurrence rate at 2 years: 35%
"""

print("\nStep 3: Adding table data...")
print("-"*40)
# In practice, uncomment the following:
# final_summary = refine_summary_incrementally(
#     refined_with_figures,
#     example_table_info,
#     "Table data"
# )
# print(final_summary)
print("[LLM call would refine summary with table data here]")

print("\n" + "="*80)
print("WORKFLOW SUMMARY:")
print("-"*80)
print("""
The incremental refine approach:

1. Process text data -> Generate initial summary
2. Process figures -> Refine summary with visual findings  
3. Process tables -> Refine summary with quantitative data

Benefits:
- No need to reprocess all data when new info arrives
- Context is preserved across refinements
- Modular pipeline (text/figures/tables can be processed separately)
- Cost-effective (only new data hits the LLM in refinement step)
""")
print("="*80)

In [ ]:
import re
import hashlib
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Set, Tuple

@dataclass
class SentenceWithMetadata:
    """
    A sentence with full metadata for citation tracking AND database provenance.
    
    Database Linkage:
        pmcid: Links to documents.pmcid in the database
        text_element_id: Links to text_elements.id in the database
        section: Maps to text_elements.path_string
    
    Citation Tracking:
        id: Unique sentence identifier for LLM citations (e.g., "S1", "S2")
        position: Position in the current concept file (0-indexed)
    
    Entity Information:
        entity_text: The specific entity mention in this sentence
        entity_start_char, entity_end_char: Character positions
        umls_score: Entity linking confidence (0.0-1.0)
    
    General:
        text: The full sentence text
        char_count, word_count: Text statistics
        hash: MD5 hash for deduplication
    """
    # Citation identifier
    id: str
    text: str
    position: int
    
    # DATABASE PROVENANCE - Links back to nlp_histo database schema
    pmcid: Optional[str] = None  # -> documents.pmcid
    text_element_id: Optional[int] = None  # -> text_elements.id
    section: Optional[str] = None  # -> text_elements.path_string
    
    # Entity information
    entity_text: Optional[str] = None
    entity_start_char: Optional[int] = None
    entity_end_char: Optional[int] = None
    umls_score: Optional[float] = None
    
    # UMLS concept info
    cui: str = ""
    concept_name: str = ""
    source_file: str = ""
    
    # Computed fields
    char_count: int = 0
    word_count: int = 0
    hash: str = ""
    
    def __post_init__(self):
        self.char_count = len(self.text)
        self.word_count = len(self.text.split())
        self.hash = hashlib.md5(self.text.encode()).hexdigest()[:8]
    
    @property
    def db_reference(self) -> str:
        """Get a database reference string for this sentence."""
        if self.pmcid and self.text_element_id:
            return f"{self.pmcid}:te{self.text_element_id}"
        return f"local:{self.id}"


def prepare_sentences_with_metadata(file_data: Dict) -> List[SentenceWithMetadata]:
    """
    Convert file data into sentences with full metadata AND database provenance.
    
    Uses sentences_with_provenance if available (from JSON files),
    otherwise falls back to simple sentences list (from TXT files).
    
    Args:
        file_data: Dict from load_json_files_with_provenance() or load_text_files()
    
    Returns:
        List of SentenceWithMetadata objects with database linkage
    """
    sentences_with_meta = []
    
    # Check if we have provenance data (JSON source)
    if 'sentences_with_provenance' in file_data and file_data['sentences_with_provenance']:
        # Full provenance from JSON files
        for idx, sent_data in enumerate(file_data['sentences_with_provenance']):
            sentence = SentenceWithMetadata(
                id=f"S{idx + 1}",
                text=sent_data['sentence'].strip() if sent_data.get('sentence') else '',
                position=idx,
                # Database provenance
                pmcid=sent_data.get('pmcid'),
                text_element_id=sent_data.get('text_element_id'),
                section=sent_data.get('section'),
                # Entity info
                entity_text=sent_data.get('entity_text'),
                entity_start_char=sent_data.get('start_char'),
                entity_end_char=sent_data.get('end_char'),
                umls_score=sent_data.get('umls_score'),
                # UMLS concept
                source_file=file_data['filename'],
                cui=file_data['cui'],
                concept_name=file_data['concept_name']
            )
            sentences_with_meta.append(sentence)
    else:
        # Fallback for TXT files (limited provenance)
        for idx, text in enumerate(file_data.get('sentences', [])):
            sentence = SentenceWithMetadata(
                id=f"S{idx + 1}",
                text=text.strip(),
                position=idx,
                source_file=file_data.get('filename', ''),
                cui=file_data.get('cui', ''),
                concept_name=file_data.get('concept_name', '')
            )
            sentences_with_meta.append(sentence)
    
    return sentences_with_meta


def format_sentences_for_llm(sentences: List[SentenceWithMetadata], max_chars: int = 6000) -> str:
    """
    Format sentences with IDs for LLM input, including PMCID provenance.
    
    Each sentence is prefixed with its ID and PMCID in brackets, e.g.:
    [S1|PMC8008320] The tumor showed trabecular growth pattern...
    [S2|PMC1234567] Immunohistochemistry revealed CD34 positivity...
    
    Args:
        sentences: List of SentenceWithMetadata
        max_chars: Maximum characters to include (truncates if exceeded)
    
    Returns:
        Formatted string with sentence IDs and PMCID provenance
    """
    formatted_lines = []
    total_chars = 0
    
    for sentence in sentences:
        # Include PMCID in citation ID if available
        if sentence.pmcid:
            cite_id = f"{sentence.id}|{sentence.pmcid}"
        else:
            cite_id = sentence.id
        
        line = f"[{cite_id}] {sentence.text}"
        
        if total_chars + len(line) > max_chars:
            formatted_lines.append(f"\n[... {len(sentences) - len(formatted_lines)} more sentences truncated ...]")
            break
        
        formatted_lines.append(line)
        total_chars += len(line) + 2  # +2 for newlines
    
    return "\n\n".join(formatted_lines)


# Test with sample data
print("Citation-aware data structures with DATABASE PROVENANCE defined")
print("\nExample SentenceWithMetadata:")
example_sentence = SentenceWithMetadata(
    id="S1",
    text="Hepatocellular carcinoma shows trabecular growth pattern with increased N/C ratio.",
    position=0,
    pmcid="PMC8008320",  # -> documents.pmcid
    text_element_id=810,  # -> text_elements.id
    section="2. Case Report",  # -> text_elements.path_string
    entity_text="carcinoma",
    umls_score=0.95,
    source_file="C0019204_Hepatocellular_carcinoma.json",
    cui="C0019204",
    concept_name="Hepatocellular carcinoma"
)
print(f"  ID: {example_sentence.id}")
print(f"  Text: {example_sentence.text[:60]}...")
print(f"  DB Reference: {example_sentence.db_reference}")
print(f"  PMCID: {example_sentence.pmcid} -> documents.pmcid")
print(f"  Text Element ID: {example_sentence.text_element_id} -> text_elements.id")
print(f"  Section: {example_sentence.section}")
print(f"  UMLS Score: {example_sentence.umls_score}")

In [ ]:
import re
import hashlib
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Set, Tuple

@dataclass
class SentenceWithMetadata:
    """
    A sentence with full metadata for citation tracking.
    
    Attributes:
        id: Unique sentence identifier (e.g., "S1", "S2")
        text: The sentence text
        position: Position in the original document (0-indexed)
        char_count: Character count
        word_count: Word count
        hash: MD5 hash of text for deduplication
        source_file: Original source file
        cui: UMLS CUI of the concept
        concept_name: Name of the medical concept
    """
    id: str
    text: str
    position: int
    char_count: int = 0
    word_count: int = 0
    hash: str = ""
    source_file: str = ""
    cui: str = ""
    concept_name: str = ""
    
    def __post_init__(self):
        self.char_count = len(self.text)
        self.word_count = len(self.text.split())
        self.hash = hashlib.md5(self.text.encode()).hexdigest()[:8]


def prepare_sentences_with_metadata(file_data: Dict) -> List[SentenceWithMetadata]:
    """
    Convert file data into sentences with full metadata.
    
    Args:
        file_data: Dict from load_text_files() containing sentences and metadata
    
    Returns:
        List of SentenceWithMetadata objects
    """
    sentences_with_meta = []
    
    for idx, text in enumerate(file_data['sentences']):
        sentence = SentenceWithMetadata(
            id=f"S{idx + 1}",
            text=text.strip(),
            position=idx,
            source_file=file_data['filename'],
            cui=file_data['cui'],
            concept_name=file_data['concept_name']
        )
        sentences_with_meta.append(sentence)
    
    return sentences_with_meta


def format_sentences_for_llm(sentences: List[SentenceWithMetadata], max_chars: int = 6000) -> str:
    """
    Format sentences with IDs for LLM input.
    
    Each sentence is prefixed with its ID in brackets, e.g.:
    [S1] The tumor showed trabecular growth pattern...
    [S2] Immunohistochemistry revealed CD34 positivity...
    
    Args:
        sentences: List of SentenceWithMetadata
        max_chars: Maximum characters to include (truncates if exceeded)
    
    Returns:
        Formatted string with sentence IDs
    """
    formatted_lines = []
    total_chars = 0
    
    for sentence in sentences:
        line = f"[{sentence.id}] {sentence.text}"
        
        if total_chars + len(line) > max_chars:
            formatted_lines.append(f"\n[... {len(sentences) - len(formatted_lines)} more sentences truncated ...]")
            break
        
        formatted_lines.append(line)
        total_chars += len(line) + 2  # +2 for newlines
    
    return "\n\n".join(formatted_lines)


# Test with sample data
print("Citation-aware data structures defined")
print("\nExample SentenceWithMetadata:")
example_sentence = SentenceWithMetadata(
    id="S1",
    text="Hepatocellular carcinoma shows trabecular growth pattern with increased N/C ratio.",
    position=0,
    source_file="C0019204_Hepatocellular_carcinoma.txt",
    cui="C0019204",
    concept_name="Hepatocellular carcinoma"
)
print(f"  ID: {example_sentence.id}")
print(f"  Text: {example_sentence.text[:60]}...")
print(f"  Word count: {example_sentence.word_count}")
print(f"  Hash: {example_sentence.hash}")

In [ ]:
# Citation-aware summarization prompt with DATABASE PROVENANCE
CITATION_SUMMARIZATION_PROMPT = PromptTemplate(
    input_variables=["concept_name", "text"],
    template="""You are a medical expert analyzing histopathology literature with rigorous citation practices.

Concept: {concept_name}

Source Sentences (each prefixed with an ID and PMCID like [S1|PMC1234567]):
{text}

Task: Create a comprehensive summary of the key medical findings. 

IMPORTANT CITATION RULES:
1. EVERY factual claim in your summary MUST include inline citations
2. Use the EXACT citation format provided: [S1|PMC1234567] or [S2|PMC7890123]
3. The PMCID links to the source paper in the database
4. Multiple sources can be cited together: [S1|PMC111, S3|PMC222]
5. Do NOT make any claims without citations
6. If you cannot find a source for a claim, do NOT include that claim

Focus on:
- Clinical significance and diagnostic criteria
- Histopathological features and characteristics
- Treatment approaches and outcomes
- Important associations or risk factors

Example format:
"Hepatocellular carcinoma typically shows trabecular growth patterns [S1|PMC8008320, S3|PMC7654321] with increased nuclear-to-cytoplasmic ratio [S2|PMC8008320]."

Summary with citations:"""
)

# Citation-aware refine prompt for incremental updates with DATABASE PROVENANCE
CITATION_REFINE_PROMPT = PromptTemplate(
    input_variables=["existing_summary", "new_sentences", "info_type"],
    template="""You are a medical expert refining a summary with new information while maintaining citations.

Existing Summary (with citations):
{existing_summary}

New {info_type} (each prefixed with an ID and PMCID):
{new_sentences}

Task: Refine the summary by incorporating the new information.

CITATION RULES:
1. Keep all existing citations from the original summary
2. Add new citations from the new sentences where applicable
3. Use the EXACT IDs provided (e.g., [S15|PMC1234567], [F1], [T1])
4. Every claim must have at least one citation
5. If new info contradicts existing info, cite both sources

Refined Summary with citations:"""
)

# Create citation-aware chain
citation_summarization_chain = LLMChain(llm=llm, prompt=CITATION_SUMMARIZATION_PROMPT)

print("Citation-aware prompts with DATABASE PROVENANCE defined")
print("\nCitation format: [S1|PMC1234567]")
print("  - S1: Local sentence ID for tracking")
print("  - PMC1234567: Links to documents.pmcid in database")
print("\nThis enables full traceability: Summary claim -> Citation -> PMCID -> Database")

In [ ]:
# Citation-aware summarization prompt
CITATION_SUMMARIZATION_PROMPT = PromptTemplate(
    input_variables=["concept_name", "text"],
    template="""You are a medical expert analyzing histopathology literature with rigorous citation practices.

Concept: {concept_name}

Source Sentences (each prefixed with an ID like [S1], [S2], etc.):
{text}

Task: Create a comprehensive summary of the key medical findings. 

IMPORTANT CITATION RULES:
1. EVERY factual claim in your summary MUST include inline citations
2. Use the format [S1], [S2], etc. to cite the source sentence(s)
3. Multiple sources can be cited together: [S1, S3, S7]
4. Do NOT make any claims without citations
5. If you cannot find a source for a claim, do NOT include that claim

Focus on:
- Clinical significance and diagnostic criteria
- Histopathological features and characteristics
- Treatment approaches and outcomes
- Important associations or risk factors

Example format:
"Hepatocellular carcinoma typically shows trabecular growth patterns [S1, S3] with increased nuclear-to-cytoplasmic ratio [S2]. CD34 is positive in sinusoidal endothelium [S5]."

Summary with citations:"""
)

# Citation-aware refine prompt for incremental updates
CITATION_REFINE_PROMPT = PromptTemplate(
    input_variables=["existing_summary", "new_sentences", "info_type"],
    template="""You are a medical expert refining a summary with new information while maintaining citations.

Existing Summary (with citations):
{existing_summary}

New {info_type} (each prefixed with an ID):
{new_sentences}

Task: Refine the summary by incorporating the new information.

CITATION RULES:
1. Keep all existing citations from the original summary
2. Add new citations from the new sentences where applicable
3. Use the exact IDs provided (e.g., [S15], [F1], [T1])
4. Every claim must have at least one citation
5. If new info contradicts existing info, cite both sources

Refined Summary with citations:"""
)

# Create citation-aware chain
citation_summarization_chain = LLMChain(llm=llm, prompt=CITATION_SUMMARIZATION_PROMPT)

print("Citation-aware prompts defined")
print("\nKey features:")
print("  - Every claim must have inline citations [S1], [S2], etc.")
print("  - Multiple citations can be combined: [S1, S3, S7]")
print("  - No uncited claims allowed")

In [ ]:
@dataclass
class CitationAnalysis:
    """
    Analysis results for citations in a summary with DATABASE PROVENANCE.
    
    Attributes:
        cited_ids: Set of sentence IDs that were cited (e.g., {"S1", "S2"})
        cited_pmcids: Set of PMCIDs that were cited (database linkage)
        valid_citations: Citations that match valid source IDs
        invalid_citations: Citations that don't match any source
        source_coverage: Percentage of source sentences that were cited
        document_coverage: Percentage of source documents (PMCIDs) cited
        hallucination_risk: % of summary sentences without citations
    """
    cited_ids: Set[str]
    cited_pmcids: Set[str]  # Database linkage
    valid_citations: List[str]
    invalid_citations: List[str]
    total_citations: int
    source_coverage: float  # % of source sentences cited
    document_coverage: float  # % of source documents cited
    hallucination_risk: float  # % of summary sentences without citations
    

def extract_citations(text: str) -> List[Dict[str, str]]:
    """
    Extract all citations from text, including PMCID provenance.
    
    Handles formats like:
    - [S1|PMC8008320]
    - [S1|PMC111, S2|PMC222]
    - [S1] (legacy format without PMCID)
    - [F1], [T1] (figures/tables)
    
    Returns:
        List of dicts with 'id' and 'pmcid' keys
    """
    # Pattern matches [S1|PMC123] or [S1, S2|PMC456] etc.
    bracket_pattern = r'\[([^\]]+)\]'
    matches = re.findall(bracket_pattern, text)
    
    all_citations = []
    for match in matches:
        # Split by comma for multiple citations
        parts = [p.strip() for p in match.split(',')]
        
        for part in parts:
            citation = {'id': '', 'pmcid': None}
            
            if '|' in part:
                # Format: S1|PMC8008320
                id_part, pmcid_part = part.split('|', 1)
                citation['id'] = id_part.strip()
                citation['pmcid'] = pmcid_part.strip()
            else:
                # Legacy format: S1 (no PMCID)
                citation['id'] = part.strip()
            
            all_citations.append(citation)
    
    return all_citations


def analyze_citations(
    summary: str,
    source_sentences: List[SentenceWithMetadata]
) -> CitationAnalysis:
    """
    Analyze citations in a summary against source sentences WITH DATABASE LINKAGE.
    
    Args:
        summary: The generated summary with citations
        source_sentences: List of source sentences with metadata
    
    Returns:
        CitationAnalysis with metrics including database coverage
    """
    # Get valid source IDs and PMCIDs
    valid_source_ids = {s.id for s in source_sentences}
    valid_pmcids = {s.pmcid for s in source_sentences if s.pmcid}
    
    # Extract citations from summary
    citations = extract_citations(summary)
    cited_ids = {c['id'] for c in citations}
    cited_pmcids = {c['pmcid'] for c in citations if c['pmcid']}
    
    # Categorize citations
    valid_citations = [c['id'] for c in citations if c['id'] in valid_source_ids]
    invalid_citations = [c['id'] for c in citations if c['id'] not in valid_source_ids]
    
    # Calculate source coverage (what % of source sentences were used)
    source_coverage = len(cited_ids & valid_source_ids) / len(valid_source_ids) * 100 if valid_source_ids else 0
    
    # Calculate document coverage (what % of source papers were cited)
    document_coverage = len(cited_pmcids & valid_pmcids) / len(valid_pmcids) * 100 if valid_pmcids else 0
    
    # Estimate hallucination risk by looking for sentences without citations
    summary_sentences = [s.strip() for s in re.split(r'[.!?]', summary) if s.strip()]
    uncited_sentences = [s for s in summary_sentences if not re.search(r'\[S\d+', s)]
    hallucination_risk = len(uncited_sentences) / len(summary_sentences) * 100 if summary_sentences else 0
    
    return CitationAnalysis(
        cited_ids=cited_ids,
        cited_pmcids=cited_pmcids,
        valid_citations=valid_citations,
        invalid_citations=invalid_citations,
        total_citations=len(citations),
        source_coverage=source_coverage,
        document_coverage=document_coverage,
        hallucination_risk=hallucination_risk
    )


def get_cited_source_text(
    summary: str,
    source_sentences: List[SentenceWithMetadata]
) -> Dict[str, Dict]:
    """
    Get a mapping of citation IDs to their source text AND database reference.
    
    Returns:
        Dict mapping citation ID to {'text': ..., 'pmcid': ..., 'text_element_id': ...}
    """
    # Build lookup
    id_to_sentence = {s.id: s for s in source_sentences}
    
    # Extract citations
    citations = extract_citations(summary)
    
    # Map cited IDs to their full info
    cited_sources = {}
    for citation in citations:
        citation_id = citation['id']
        if citation_id in id_to_sentence:
            sent = id_to_sentence[citation_id]
            cited_sources[citation_id] = {
                'text': sent.text,
                'pmcid': sent.pmcid,
                'text_element_id': sent.text_element_id,
                'section': sent.section,
                'db_reference': sent.db_reference
            }
        else:
            cited_sources[citation_id] = {
                'text': "[INVALID - Source not found]",
                'pmcid': None,
                'text_element_id': None,
                'section': None,
                'db_reference': None
            }
    
    return cited_sources


# Test citation extraction with new format
test_summary = """
Hepatocellular carcinoma shows trabecular growth [S1|PMC8008320, S3|PMC7654321] 
with increased N/C ratio [S2|PMC8008320].
CD34 positivity is diagnostic [S5|PMC9999999]. Treatment outcomes vary [S7, S8, S9].
Some findings suggest poor prognosis [S99|PMC000].
"""

extracted = extract_citations(test_summary)
print("Citation extraction test (with PMCID provenance):")
print(f"  Input: {test_summary[:80]}...")
print(f"\n  Extracted citations:")
for c in extracted[:5]:
    print(f"    ID: {c['id']}, PMCID: {c['pmcid']}")
print(f"\n  Unique sentence IDs: {set(c['id'] for c in extracted)}")
print(f"  Unique PMCIDs: {set(c['pmcid'] for c in extracted if c['pmcid'])}")

In [ ]:
@dataclass
class CitationAnalysis:
    """
    Analysis results for citations in a summary.
    
    Attributes:
        cited_ids: Set of sentence IDs that were cited
        valid_citations: Citations that match valid source IDs
        invalid_citations: Citations that don't match any source
        uncited_claims: Sentences in summary without citations (potential hallucinations)
        source_coverage: Percentage of source sentences that were cited
    """
    cited_ids: Set[str]
    valid_citations: List[str]
    invalid_citations: List[str]
    total_citations: int
    source_coverage: float  # % of source sentences cited
    hallucination_risk: float  # % of summary sentences without citations
    

def extract_citations(text: str) -> List[str]:
    """
    Extract all citation IDs from text.
    
    Handles formats like:
    - [S1]
    - [S1, S2, S3]
    - [S1,S2]
    - [F1] (for figures)
    - [T1] (for tables)
    
    Returns:
        List of all citation IDs found (e.g., ["S1", "S2", "F1"])
    """
    # Pattern matches [S1], [S1, S2], [S1,S2,S3], etc.
    bracket_pattern = r'\[([^\]]+)\]'
    matches = re.findall(bracket_pattern, text)
    
    all_citations = []
    for match in matches:
        # Split by comma and clean each ID
        ids = [id.strip() for id in match.split(',')]
        all_citations.extend(ids)
    
    return all_citations


def analyze_citations(
    summary: str,
    source_sentences: List[SentenceWithMetadata]
) -> CitationAnalysis:
    """
    Analyze citations in a summary against source sentences.
    
    Args:
        summary: The generated summary with citations
        source_sentences: List of source sentences with metadata
    
    Returns:
        CitationAnalysis with metrics
    """
    # Get valid source IDs
    valid_source_ids = {s.id for s in source_sentences}
    
    # Extract citations from summary
    citations = extract_citations(summary)
    cited_ids = set(citations)
    
    # Categorize citations
    valid_citations = [c for c in citations if c in valid_source_ids]
    invalid_citations = [c for c in citations if c not in valid_source_ids]
    
    # Calculate coverage (what % of sources were used)
    source_coverage = len(cited_ids & valid_source_ids) / len(valid_source_ids) * 100 if valid_source_ids else 0
    
    # Estimate hallucination risk by looking for sentences without citations
    # Split summary into sentences and check each for citations
    summary_sentences = [s.strip() for s in re.split(r'[.!?]', summary) if s.strip()]
    uncited_sentences = [s for s in summary_sentences if not re.search(r'\[S\d+', s)]
    hallucination_risk = len(uncited_sentences) / len(summary_sentences) * 100 if summary_sentences else 0
    
    return CitationAnalysis(
        cited_ids=cited_ids,
        valid_citations=valid_citations,
        invalid_citations=invalid_citations,
        total_citations=len(citations),
        source_coverage=source_coverage,
        hallucination_risk=hallucination_risk
    )


def get_cited_source_text(
    summary: str,
    source_sentences: List[SentenceWithMetadata]
) -> Dict[str, str]:
    """
    Get a mapping of citation IDs to their source text.
    
    Useful for verification - allows you to see exactly what each citation refers to.
    
    Returns:
        Dict mapping citation ID to source sentence text
    """
    # Build lookup
    id_to_text = {s.id: s.text for s in source_sentences}
    
    # Extract citations
    citations = extract_citations(summary)
    
    # Map cited IDs to their text
    cited_sources = {}
    for citation_id in set(citations):
        if citation_id in id_to_text:
            cited_sources[citation_id] = id_to_text[citation_id]
        else:
            cited_sources[citation_id] = "[INVALID - Source not found]"
    
    return cited_sources


# Test citation extraction
test_summary = """
Hepatocellular carcinoma shows trabecular growth [S1, S3] with increased N/C ratio [S2].
CD34 positivity in sinusoidal endothelium is diagnostic [S5]. Treatment outcomes vary [S7, S8, S9].
Some findings suggest poor prognosis [S99].
"""

extracted = extract_citations(test_summary)
print("Citation extraction test:")
print(f"  Input: {test_summary[:80]}...")
print(f"  Extracted citations: {extracted}")
print(f"  Unique IDs: {set(extracted)}")

## Citation-Aware Processing Pipeline

Process documents with full citation tracking and quality assessment.

In [ ]:
def process_with_citations(file_data: Dict) -> Dict:
    """
    Process a document with full citation tracking.
    
    This generates a summary with inline citations and provides
    quality metrics for assessment.
    
    Args:
        file_data: Dict from load_text_files()
    
    Returns:
        Dict with summary, citations, analysis, and audit trail
    """
    # Prepare sentences with metadata
    sentences = prepare_sentences_with_metadata(file_data)
    
    # Format for LLM (with IDs)
    formatted_text = format_sentences_for_llm(sentences)
    
    # Skip if too few sentences
    if len(sentences) < 3:
        return {
            'status': 'skipped',
            'cui': file_data['cui'],
            'concept_name': file_data['concept_name'],
            'reason': 'Too few sentences'
        }
    
    try:
        # Generate summary with citations
        result = citation_summarization_chain.invoke({
            "concept_name": file_data['concept_name'],
            "text": formatted_text
        })
        summary = result['text'].strip()
        
        # Analyze citations
        analysis = analyze_citations(summary, sentences)
        
        # Get cited source texts for verification
        cited_sources = get_cited_source_text(summary, sentences)
        
        # Build comprehensive audit trail
        audit_trail = {
            'cui': file_data['cui'],
            'concept_name': file_data['concept_name'],
            'source_file': file_data['filename'],
            'num_source_sentences': len(sentences),
            'processing_timestamp': datetime.now().isoformat(),
            'summary_with_citations': summary,
            'citation_analysis': {
                'total_citations': analysis.total_citations,
                'valid_citations': analysis.valid_citations,
                'invalid_citations': analysis.invalid_citations,
                'cited_sentence_ids': list(analysis.cited_ids),
                'source_coverage_percent': round(analysis.source_coverage, 2),
                'hallucination_risk_percent': round(analysis.hallucination_risk, 2)
            },
            'cited_source_texts': cited_sources,
            'source_sentences': [asdict(s) for s in sentences]
        }
        
        return {
            'status': 'success',
            'cui': file_data['cui'],
            'concept_name': file_data['concept_name'],
            'summary': summary,
            'analysis': analysis,
            'cited_sources': cited_sources,
            'sentences': sentences,
            'audit_trail': audit_trail
        }
    
    except Exception as e:
        return {
            'status': 'error',
            'cui': file_data['cui'],
            'concept_name': file_data['concept_name'],
            'error': str(e)
        }


# Output directory for citation-aware summaries
CITATION_DIR = OUTPUT_DIR / "citation_summaries"
CITATION_DIR.mkdir(parents=True, exist_ok=True)

print("Citation-aware processing pipeline defined")
print(f"Output directory: {CITATION_DIR}")

In [ ]:
# Run citation-aware summarization on all files
citation_results = []
citation_errors = []
citation_skipped = []

print("="*80)
print("CITATION-AWARE SUMMARIZATION")
print("="*80 + "\n")

for i, file_data in enumerate(all_text_files, 1):
    cui = file_data['cui']
    concept_name = file_data['concept_name']
    
    if i % 50 == 0 or i <= 5:
        print(f"[{i}/{len(all_text_files)}] Processing {cui} - {concept_name[:40]}...")
    
    result = process_with_citations(file_data)
    
    if result['status'] == 'success':
        citation_results.append(result)
    elif result['status'] == 'skipped':
        citation_skipped.append(result)
    else:
        citation_errors.append(result)
        print(f"  Error on {cui}: {result.get('error', 'Unknown')}")

print(f"\n{'='*80}")
print(f"Citation-Aware Summarization Complete:")
print(f"  Successful: {len(citation_results)}")
print(f"  Skipped: {len(citation_skipped)}")
print(f"  Errors: {len(citation_errors)}")
print("="*80)

In [ ]:
print("="*80)
print("CITATION-AWARE SUMMARIZATION QUALITY METRICS")
print("(With Database Provenance)")
print("="*80)

if citation_results:
    # Aggregate metrics
    all_coverage = [r['analysis'].source_coverage for r in citation_results]
    all_doc_coverage = [r['analysis'].document_coverage for r in citation_results]
    all_hallucination = [r['analysis'].hallucination_risk for r in citation_results]
    all_citations = [r['analysis'].total_citations for r in citation_results]
    all_invalid = [len(r['analysis'].invalid_citations) for r in citation_results]
    
    print("\n" + "-"*40)
    print("OVERALL QUALITY STATISTICS")
    print("-"*40)
    print(f"\nProcessed: {len(citation_results)} concepts")
    
    print(f"\nSource Sentence Coverage (% of source sentences cited):")
    print(f"  Average: {sum(all_coverage)/len(all_coverage):.1f}%")
    print(f"  Min:     {min(all_coverage):.1f}%")
    print(f"  Max:     {max(all_coverage):.1f}%")
    
    print(f"\nDocument Coverage (% of source papers/PMCIDs cited):")
    print(f"  Average: {sum(all_doc_coverage)/len(all_doc_coverage):.1f}%")
    print(f"  Min:     {min(all_doc_coverage):.1f}%")
    print(f"  Max:     {max(all_doc_coverage):.1f}%")
    
    print(f"\nHallucination Risk (% of summary sentences without citations):")
    print(f"  Average: {sum(all_hallucination)/len(all_hallucination):.1f}%")
    print(f"  Min:     {min(all_hallucination):.1f}%")
    print(f"  Max:     {max(all_hallucination):.1f}%")
    
    print(f"\nCitations per Summary:")
    print(f"  Average: {sum(all_citations)/len(all_citations):.1f}")
    print(f"  Min:     {min(all_citations)}")
    print(f"  Max:     {max(all_citations)}")
    
    print(f"\nInvalid Citations (pointing to non-existent sources):")
    print(f"  Total:   {sum(all_invalid)}")
    print(f"  Average: {sum(all_invalid)/len(all_invalid):.2f} per summary")
    
    # Quality categorization
    high_quality = [r for r in citation_results 
                    if r['analysis'].hallucination_risk < 20 and r['analysis'].source_coverage > 10]
    medium_quality = [r for r in citation_results 
                      if 20 <= r['analysis'].hallucination_risk < 50]
    low_quality = [r for r in citation_results 
                   if r['analysis'].hallucination_risk >= 50]
    
    print("\n" + "-"*40)
    print("QUALITY DISTRIBUTION")
    print("-"*40)
    print(f"  High quality (hallucination <20%, coverage >10%): {len(high_quality)}")
    print(f"  Medium quality (hallucination 20-50%):           {len(medium_quality)}")
    print(f"  Low quality (hallucination >=50%):               {len(low_quality)}")
    
    # Show sample with citation verification
    print("\n" + "-"*40)
    print("SAMPLE CITATION-AWARE SUMMARY (With DB Provenance)")
    print("-"*40)
    
    sample = citation_results[0]
    print(f"\nConcept: {sample['concept_name']}")
    print(f"CUI: {sample['cui']}")
    print(f"\nSummary with citations:")
    print(sample['summary'][:600] + "..." if len(sample['summary']) > 600 else sample['summary'])
    
    print(f"\nCitation Analysis:")
    print(f"  Total citations: {sample['analysis'].total_citations}")
    print(f"  Source sentence coverage: {sample['analysis'].source_coverage:.1f}%")
    print(f"  Document (PMCID) coverage: {sample['analysis'].document_coverage:.1f}%")
    print(f"  Hallucination risk: {sample['analysis'].hallucination_risk:.1f}%")
    print(f"  PMCIDs cited: {sample['analysis'].cited_pmcids}")
    print(f"  Invalid citations: {sample['analysis'].invalid_citations}")
    
    print("\n" + "-"*40)
    print("CITED SOURCES WITH DATABASE REFERENCE (First 3)")
    print("-"*40)
    for i, (cite_id, info) in enumerate(list(sample['cited_sources'].items())[:3], 1):
        text = info['text'][:100] + "..." if len(info['text']) > 100 else info['text']
        print(f"\n[{cite_id}]:")
        print(f"  Text: {text}")
        print(f"  PMCID: {info['pmcid']} -> documents.pmcid")
        print(f"  Text Element ID: {info['text_element_id']} -> text_elements.id")
        print(f"  Section: {info['section']}")
        print(f"  DB Reference: {info['db_reference']}")

else:
    print("\nNo citation-aware results to analyze.")

print("\n" + "="*80)
print("DATABASE LINKAGE SUMMARY")
print("-"*80)
print("""
Each citation [S1|PMC1234567] links to:
  - PMC1234567 -> SELECT * FROM documents WHERE pmcid = 'PMC1234567'
  - text_element_id -> SELECT * FROM text_elements WHERE id = <id>
  
Full query to verify a citation:
  SELECT d.pmcid, d.title, te.text_content, te.path_string
  FROM documents d
  JOIN text_elements te ON d.id = te.document_id
  WHERE d.pmcid = 'PMC1234567' AND te.id = <text_element_id>
""")
print("="*80)

In [ ]:
print("="*80)
print("CITATION-AWARE SUMMARIZATION QUALITY METRICS")
print("="*80)

if citation_results:
    # Aggregate metrics
    all_coverage = [r['analysis'].source_coverage for r in citation_results]
    all_hallucination = [r['analysis'].hallucination_risk for r in citation_results]
    all_citations = [r['analysis'].total_citations for r in citation_results]
    all_invalid = [len(r['analysis'].invalid_citations) for r in citation_results]
    
    print("\n" + "-"*40)
    print("OVERALL QUALITY STATISTICS")
    print("-"*40)
    print(f"\nProcessed: {len(citation_results)} concepts")
    print(f"\nSource Coverage (% of source sentences cited):")
    print(f"  Average: {sum(all_coverage)/len(all_coverage):.1f}%")
    print(f"  Min:     {min(all_coverage):.1f}%")
    print(f"  Max:     {max(all_coverage):.1f}%")
    
    print(f"\nHallucination Risk (% of summary sentences without citations):")
    print(f"  Average: {sum(all_hallucination)/len(all_hallucination):.1f}%")
    print(f"  Min:     {min(all_hallucination):.1f}%")
    print(f"  Max:     {max(all_hallucination):.1f}%")
    
    print(f"\nCitations per Summary:")
    print(f"  Average: {sum(all_citations)/len(all_citations):.1f}")
    print(f"  Min:     {min(all_citations)}")
    print(f"  Max:     {max(all_citations)}")
    
    print(f"\nInvalid Citations (pointing to non-existent sources):")
    print(f"  Total:   {sum(all_invalid)}")
    print(f"  Average: {sum(all_invalid)/len(all_invalid):.2f} per summary")
    
    # Quality categorization
    high_quality = [r for r in citation_results if r['analysis'].hallucination_risk < 20 and r['analysis'].source_coverage > 10]
    medium_quality = [r for r in citation_results if 20 <= r['analysis'].hallucination_risk < 50]
    low_quality = [r for r in citation_results if r['analysis'].hallucination_risk >= 50]
    
    print("\n" + "-"*40)
    print("QUALITY DISTRIBUTION")
    print("-"*40)
    print(f"  High quality (hallucination <20%, coverage >10%): {len(high_quality)}")
    print(f"  Medium quality (hallucination 20-50%):           {len(medium_quality)}")
    print(f"  Low quality (hallucination >=50%):               {len(low_quality)}")
    
    # Show sample with citation verification
    print("\n" + "-"*40)
    print("SAMPLE CITATION-AWARE SUMMARY")
    print("-"*40)
    
    sample = citation_results[0]
    print(f"\nConcept: {sample['concept_name']}")
    print(f"CUI: {sample['cui']}")
    print(f"\nSummary with citations:")
    print(sample['summary'][:600] + "..." if len(sample['summary']) > 600 else sample['summary'])
    
    print(f"\nCitation Analysis:")
    print(f"  Total citations: {sample['analysis'].total_citations}")
    print(f"  Source coverage: {sample['analysis'].source_coverage:.1f}%")
    print(f"  Hallucination risk: {sample['analysis'].hallucination_risk:.1f}%")
    print(f"  Invalid citations: {sample['analysis'].invalid_citations}")
    
    print("\n" + "-"*40)
    print("CITED SOURCES (First 3)")
    print("-"*40)
    for i, (cite_id, text) in enumerate(list(sample['cited_sources'].items())[:3], 1):
        print(f"\n[{cite_id}]: {text[:150]}..." if len(text) > 150 else f"\n[{cite_id}]: {text}")

else:
    print("\nNo citation-aware results to analyze.")

print("\n" + "="*80)
print("QUALITY ASSESSMENT COMPLETE")
print("="*80)